In [22]:
from utils import * 
from files.pdb import PDBFile
import numpy as np
from typing import NamedTuple
from dataclasses import dataclass, asdict
import shutil 
from matplotlib.lines import Line2D
from datetime import datetime
import itertools
import os
import re
import orjson 

%load_ext autoreload 
%autoreload 2

date = datetime.now().strftime("%m_%d_%Y")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
ALPHAFOLD_INPUT_DIR = '../data/genes/alphafold/af_input' # Directory name to match what is required on biotite.
ALPHAFOLD_OUTPUT_DIR = '../data/genes/alphafold/af_output' # Directory name to match what is required on biotite.

MMSEQS_DIR = '../data/genes/mmseqs'
MMSEQS_DATABASE_DIR = '../data/genes/mmseqs/db'
MMSEQS_TMP_DIR = '../data/tmp'

In [24]:
level_1_conserved_cluster_ids = [8, 7, 0, 4, 5, 6, 1, 3, 2]

In [25]:
# Interested in looking at potential protein-protein interactions, particularly potential interactions with the ATPase as a possible hint at function. 

genes_df = pd.read_csv('../data/genes/genes.csv', index_col=0)
genes_df['cluster_id'] = genes_df.index.map(json.load(open('../data/genes/clusters.json', 'r')))
genes_df = genes_df[genes_df.genome_id != 'bz_12'].copy()
genes_df['alphafold_id'] = genes_df.apply(lambda row : f'{row.genome_id}_cluster_{int(row.cluster_id)}_1mer', axis=1)
genes_df = genes_df[genes_df.cluster_id.isin(level_1_conserved_cluster_ids)].copy()

FASTAFile.from_df(genes_df).write('../data/genes/level_1_conserved_clusters.faa')

In [26]:
# names = list()

# for genome_id, df in genes_df.groupby('genome_id'):
#     input = list()
#     for pair in itertools.combinations(df.to_dict(orient='records'), 2):
#         name = pair[0]['alphafold_id'] + '-' + pair[1]['alphafold_id']
#         names.append(name)
        
#         file = AlphaFoldInputFile(name)
#         file.add_seq(pair[0]['seq'])
#         file.add_seq(pair[1]['seq'])

#         input += file.get_info()

#     file_name = f'{genome_id}_level_1_clusters_ppi.json'
#     with open(os.path.join(ALPHAFOLD_INPUT_DIR, file_name), 'w') as f:
#         json.dump(input, f)
#     print(ALPHAFOLD_BIOTITE_CMD.format(file_name=file_name))

# # https://app.gitbook.com/o/-LzcB3BNVSNh_20MBLKi/s/-M-S5z_vnCqDzDHcfsmi/protein-folding

# output_dirs = [os.path.join(ALPHAFOLD_OUTPUT_DIR, name) for name in names]
# # assert np.all([os.path.isdir(path) for path in output_dirs]), 'Missing some of the expected output directories.'


In [27]:


def get_msas_unpaired(gene_ids:list, dir_path:str=MMSEQS_DIR):
    '''Load the unpaired MSAs for the specified gene IDs.
    
    :param gene_ids: The IDs of the genes to construct paired MSAs for. These should all be from the same genome.   
    :param dir_path: The path to the directory where the MSAs are stored. This function assumes the a3m files have names matching the gene ID of the query 
        sequence. 
    '''
    paths = {gene_id:os.path.join(dir_path, f'{gene_id}.a3m') for gene_id in gene_ids}
    return {gene_id:str(FASTAFile.from_file(path)) for gene_id, path in paths.items()}


def get_msas_paired(gene_ids:list, dir_path:str=MMSEQS_DIR):
    '''Construct paired MSAs using the specified gene IDs as queries. This is done by:
        (1) Loading the unpaired MSA for each query gene ID. 
        (2) Filtering each unpaired MSA to include only genes from genomes which are represented in both, i.e. contain identified homologs of each query sequence. 
        (3) Sorting the MSAs alphabetically by gene ID (ensuring the query is still the first entry) so that rows correspond.
        
    :param gene_ids: The IDs of the genes to construct paired MSAs for. These should all be from the same genome.   
    :param dir_path: The path to the directory where the MSAs are stored. This function assumes the a3m files have names matching the gene ID of the query 
        sequence. 
    '''
    assert len(set([get_genome_id(gene_id) for gene_id in gene_ids])) == 1, 'get_msas_paired: Expected all genes to be from the same genome.'

    paths = {gene_id:os.path.join(dir_path, f'{gene_id}.a3m') for gene_id in gene_ids}
    msas = {gene_id:FASTAFile.from_file(path).to_df().reset_index(names='gene_id') for gene_id, path in paths.items()}
    msas = {gene_id:df.assign(genome_id=df.gene_id.apply(get_genome_id)) for gene_id, df in msas.items()}

    genome_ids = None
    for _, df in msas.items(): # Get all genome IDs which have a representative for all chains. 
        genome_ids = df.genome_id.unique() if (genome_ids is None) else np.intersect1d(genome_ids, df.genome_id.unique())
    
    if len(genome_ids) < 2:
        # If there are two few genome IDs, the paired MSA will just contain the genes from the query genome (but do not want it to throw an error). 
        print('get_msas_paired: Too few paired sequences for a proper paired MSA.')

    sort = lambda df : pd.concat([df.iloc[:1], df.iloc[1:].sort_values('gene_id')]) # Sort a DataFrame by genome ID, while holding the query row fixed.
    msas = {gene_id:sort(df[df.genome_id.isin(genome_ids)].copy()) for gene_id, df in msas.items()}
    
    return {gene_id:str(FASTAFile.from_df(df.set_index('gene_id'))) for gene_id, df in msas.items()}


# make_msas('../data/genes/level_1_conserved_clusters.faa' )

# Clean up the created databases and folders.
for path in glob.glob(os.path.join(MMSEQS_DATABASE_DIR, '*')): 
    os.remove(path)
shutil.rmtree(MMSEQS_TMP_DIR, ignore_errors=True)


    

In [28]:
for genome_id, df in genes_df.groupby('genome_id'):

    for pair in itertools.combinations(df.reset_index(names='gene_id').to_dict(orient='records'), 2):

        alphafold_ids = [ pair[0]['alphafold_id'],  pair[1]['alphafold_id']]
        gene_ids = [pair[0]['gene_id'], pair[1]['gene_id']]
        seqs = [pair[0]['seq'], pair[1]['seq']]

        name = '-'.join(alphafold_ids)
        names.append(name)

        msas = dict()
        msas['paired'] = get_msas_paired(gene_ids)
        msas['unpaired'] = get_msas_unpaired(gene_ids)

        file = AlphaFoldInputFile(name, dialect='alphafold3', num_seeds=5, version=4)

        for gene_id, alphafold_id, seq in zip(gene_ids, alphafold_ids, seqs):
            file.add_seq(seq, paired_msa=msas['paired'][gene_id], unpaired_msa=msas['unpaired'][gene_id], description=alphafold_id)

        file.write(os.path.join(ALPHAFOLD_INPUT_DIR, f'{name}.json'))

cmd = f'sbatch --partition gpu --gpus 1 --wrap "run_alphafold3 --input_dir /root/af_input/{os.path.basename(ALPHAFOLD_INPUT_DIR)} --model_dir=/root/models --db_dir=/root/public_databases --output_dir=/root/af_output"'
print(cmd)

# https://app.gitbook.com/o/-LzcB3BNVSNh_20MBLKi/s/-M-S5z_vnCqDzDHcfsmi/protein-folding

output_dirs = [os.path.join(ALPHAFOLD_OUTPUT_DIR, name) for name in names]
output_dirs = [path for path in output_dirs if os.path.exists(path)]


sbatch --partition gpu --gpus 1 --wrap "run_alphafold3 --input_dir /root/af_input/af_input --model_dir=/root/models --db_dir=/root/public_databases --output_dir=/root/af_output"


In [29]:
if not os.path.exists('analysis-1-ppi_data.json'):
    
    outputs = [AlphaFoldOutput(path) for path in output_dirs if os.path.exists(path)]

    data = list()
    for output in outputs:
        contact_probs_df = output.get_contact_probs(models=[output.best_model])
        paes_df =  output.get_paes(models=[output.best_model])
        chain_ids = ['A', 'B']

        idxs, idxs_shifted = get_interface_idxs(contact_probs_df, chain_ids=chain_ids, shift=True)
        idxs = [idxs[chain_ids[0]], idxs[chain_ids[1]]]

        data_ = {'name':output.name}
        data_['iptm'] = np.mean(list(output.get_iptms().values()))
        data_['idxs'] = idxs
        data_['idxs_shifted'] = [idxs_shifted[chain_ids[0]], idxs_shifted[chain_ids[1]]]
        data_['num_contacts'] = len(idxs[0])
        data_['pae'] = [paes_df.iloc[i, j] for i, j in zip(*idxs)]
        data_['contact_probs'] = [contact_probs_df.iloc[i, j] for i, j in zip(*idxs)]
        data.append(data_)

    with open('analysis-1-ppi_data.json', 'wb') as f:
        f.write(orjson.dumps(data, default=default))

else:
    with open('analysis-1-ppi_data.json', 'r') as f:
        data = json.load(f)

data = [data_ for data_ in data if (data_['num_contacts'] > 0)]
print(f'Loaded data for {len(data)} AlphaFold co-folds with predicted contacts.')


Loaded data for 75 AlphaFold co-folds with predicted contacts.


In [ ]:
# Jack said that looking at the ipTM is not super informative; there could be a valid protein-protein interaction, but low ipTM due to disordered or low-
# confidence regions. Instead, it would be worth looking at the PAE for residues on different monomers. Any error < 10 A could be worth considering as a potential contact. 

# What would be a good potential statistic for evaluating the likelihood of a protein-protein contact? Want to evaluate both (1) predicted distance and (2) PAE. 
# There is also an output called contact_probs, which is based on a seperate output head in the model (i.e. not the pLDDT or PAE head). This is discussed in the 
# supplement of Jumper et. al. 2021. 

# TODO: Take some notes on this, I don't know if I completely understand. 
# Why isn't PAE symmetric? Asymmetry can arise due to differences in the stability of the selected reference frame. 
# The PAE asks "Asuming this residue is correctly located, how wrong is the other one likely to be?"

In [ ]:
# output = AlphaFoldOutput('../data/genes/alphafold/af_output/bz_0_cluster_3_6mer-mg_atp')
# contact_probs_df = output.get_contact_probs(models=[output.best_model])
get_interface_idxs(contact_probs_df, chain_ids=['A', 'B'], min_contact_prob=0.5)

get_interface: Found 22 residues at the interface of chains A and B.


{'A': array([ 18,  18,  21,  21,  21,  23, 113, 116, 117, 117, 117, 118, 120,
        120, 121, 121, 123, 142, 142, 145, 183, 184]),
 'B': array([158, 159, 159, 160, 161,  31,  97, 131,  94,  97, 207, 207,  31,
         32, 205, 207, 258, 132, 133,  30,  30, 158])}

In [ ]:
ppis = list()
for path in output_dirs:
    cluster_ids = sorted(re.findall(r'cluster_\d+', path))
    assert len(cluster_ids) == 2, f'Only expected 2 gene clusters per file, but got {len(cluster_ids)}.'
    ppis.append('-'.join(cluster_ids))

ppis, counts = np.unique(ppis, return_counts=True)
for ppi, count in zip(ppis, counts):
    print(ppi, count)

# Would be useful to look at the confidence of the residue positions at the interface, and also whether or not the cluster_1-cluster_3 interactions affect the multimerization. 

cluster_0-cluster_2 1
cluster_0-cluster_3 1
cluster_0-cluster_5 1
cluster_0-cluster_7 10
cluster_0-cluster_8 1
cluster_1-cluster_3 11
cluster_1-cluster_4 2
cluster_2-cluster_3 1
cluster_2-cluster_5 1
cluster_2-cluster_6 1
cluster_3-cluster_4 2
cluster_3-cluster_6 2
cluster_3-cluster_7 2
cluster_4-cluster_5 10
cluster_4-cluster_6 2
cluster_4-cluster_7 1
cluster_5-cluster_6 3
cluster_5-cluster_7 1


In [ ]:
names = list()
ALPHAFOLD_INPUT_DIR = '../data/genes/alphafold/af_input/'
ALPHAFOLD_OUTPUT_DIR = '../data/genes/alphafold/af_output/'

for genome_id, df in genes_df[genes_df.cluster_id.isin([1, 3])].groupby('genome_id'):

    for pair in itertools.combinations(df.reset_index(names='gene_id').to_dict(orient='records'), 2):
        name = pair[0]['alphafold_id'].replace('1mer', '6mer') + '-' + pair[1]['alphafold_id'].replace('1mer', '6mer') + '-mg_atp'
        names.append(name)

        gene_ids = [pair[0]['gene_id'], pair[1]['gene_id']]
        seqs = [pair[0]['seq'], pair[1]['seq']]

        msas = dict()
        msas['paired'] = get_msas_paired(gene_ids)
        msas['unpaired'] = get_msas_unpaired(gene_ids)

        file = AlphaFoldInputFile(name, dialect='alphafoldserver', num_seeds=1, version=1)

        for gene_id, seq in zip(gene_ids, seqs):
            file.add_seq(seq, paired_msa=msas['paired'][gene_id], unpaired_msa=msas['unpaired'][gene_id], n=6)
        file.add_atp(n=6)
        file.add_mg(n=6)

        file.write(os.path.join(ALPHAFOLD_INPUT_DIR, f'{name}.json'))



get_interface: Found 7 residues at the interface of chains A and B.
get_interface: Found 7 residues at the interface of chains A and B.
get_interface: Found 6 residues at the interface of chains B and C.


[array([ 21, 117, 117, 120, 121, 183]), array([159,  94,  97,  31, 207,  30])]

In [ ]:
# It seems as though the cluster_1-cluster_3 contact is only predicted highly with the cluster_3 monomer. But is this a failing of AlphaFold?
# Do the contacting residues differ between the monomer predictions and the multimer predictions (i.e. does cluster_3 multimerizing block 
# the preferred interface for cluster_1 and cluster_3)?